# Phân loại hoa Iris

## Mục tiêu bài toán
Sử dụng các đặc trưng về đài hoa và cánh hoa (chiều dài, chiều rộng) để phân loại chính xác giống hoa Iris (Setosa, Versicolor, Virginica). 
- **Input:** `SepalLength`, `SepalWidth`, `PetalLength`, `PetalWidth`
- **Output:** `Class` (giống hoa)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 1. Đọc dữ liệu
Data được đọc từ thư mục `data/iris.csv`. Vì file không có header, ta sẽ định nghĩa tên các cột trước thông qua tham số `names`.

In [ ]:
# Đọc dữ liệu từ file CSV không có header
df = pd.read_csv('data/iris.csv', header=None, 
                 names=['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Class'])

display(df.head())
print(f"\nKích thước tập dữ liệu: {df.shape}")

## 2. Khám phá dữ liệu (EDA) và Xử lý Missing Values
Ta sẽ vẽ biểu đồ để xem độ phân cụm của các loài hoa (Pairplot), phân phối từng đặc trưng (Boxplot) và độ tương quan giữa các đặc trưng (Heatmap).

In [ ]:
# 2.1. Vẽ Pairplot để phân tích sự tương quan giữa các thuộc tính, phân loại theo lớp
plt.figure(figsize=(10, 8))
sns.pairplot(df, hue='Class', palette='Set2')
plt.suptitle("Phân bố các đặc trưng theo giống hoa (Pairplot)", y=1.02)
plt.show()

In [ ]:
# 2.2. Vẽ Boxplot để kiểm tra phân bố chung và dấu hiệu ngoại lai (Outliers)
plt.figure(figsize=(10, 6))
sns.boxplot(data=df.drop('Class', axis=1), palette="Set3")
plt.title("Biểu đồ Boxplot phân phối các đặc trưng hoa")
plt.show()

In [ ]:
# 2.3. Vẽ Heatmap phân tích độ tương quan 
plt.figure(figsize=(8, 6))
sns.heatmap(df.drop('Class', axis=1).corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Ma trận tương quan nhiệt độ (Heatmap)")
plt.show()

In [ ]:
# 2.4. Tiền xử lý: Xử lý Missing Values
print("Kiểm tra giá trị thiếu:")
print(df.isnull().sum())

# Ở bài toán này có thể không có giá trị rỗng, nhưng để chuẩn hóa quy trình:
# Ta tách biến X chứa các đặc trưng (chuỗi số)
X = df.drop('Class', axis=1)
y = df['Class']

# Sử dụng SimpleImputer bằng trung vị (median) thay vì dropna()
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Đưa lại thành DataFrame sau khi xử lý missing values
X = pd.DataFrame(X_imputed, columns=X.columns)

## 3. Tiền xử lý, Chia tập dữ liệu và Chuẩn hóa
Ta dùng `train_test_split` với tỷ lệ chuẩn 70/30 và `StandardScaler`.

In [ ]:
# Chia tập Train/Test theo tỷ lệ 70/30
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Chuẩn hóa dữ liệu với StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Kích thước X_train:", X_train_scaled.shape)
print("Kích thước X_test:", X_test_scaled.shape)

## 4. Huấn luyện và Đánh giá Mô hình
Sử dụng 2 thuật toán: `RandomForestClassifier` và `SVC` (Support Vector Machine). Đánh giá hiệu suất tập Test bằng `accuracy_score` và `f1_score`.

In [ ]:
# 1. Khởi tạo và huấn luyện Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_y_pred = rf_model.predict(X_test_scaled)

rf_acc = accuracy_score(y_test, rf_y_pred)
rf_f1 = f1_score(y_test, rf_y_pred, average='weighted')

# 2. Khởi tạo và huấn luyện SVC
svc_model = SVC(random_state=42)
svc_model.fit(X_train_scaled, y_train)
svc_y_pred = svc_model.predict(X_test_scaled)

svc_acc = accuracy_score(y_test, svc_y_pred)
svc_f1 = f1_score(y_test, svc_y_pred, average='weighted')

### Nhận xét Kết Quả

In [ ]:
print("=== KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ===\n")

print("1. Random Forest Classifier:")
print(f"   - Accuracy: {rf_acc:.4f}")
print(f"   - F1-Score: {rf_f1:.4f}\n")

print("2. Support Vector Classifier (SVC):")
print(f"   - Accuracy: {svc_acc:.4f}")
print(f"   - F1-Score: {svc_f1:.4f}\n")

print("-> Nhận xét: Dựa vào thông số trên, ta thấy được mô hình nào mang lại kết quả F1-Score cao hơn sẽ vượt trội hơn trên tập test phân loại hoa. (Cả 2 thường sẽ cho kết quả rất khả quan với Iris Dataset)")